In [6]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys 
import scanpy as sc
import warnings
import scanpy as sc
import anndata as ad
import os
from scipy.sparse import issparse
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
plt.rcParams["font.family"] = "Arial"
main_dir = '../'
# task_grn_inference_dir = '../../'
output_dir = f"{main_dir}/output/"

# - local imports 
from hiara import retrieve_feature_data, retrieve_sig_stats

from hiara import PRIOR_DIR, PLOTS_DIR, CLOCKS_DIR,  CELL_TYPES, surrogate_names, colors_blind, palette_cell_types, palette_datasets, palette_datasets_pretty, palette_genders, AGING_COHORTS
from hiara import retrieve_adata, retrieve_net, retrieve_net_consensus
from hiara import clock_version, use_local_clocks
from grn_benchmark.src.helper import load_env
env = load_env()
TASK_GRN_INFERENCE_DIR = env['TASK_GRN_INFERENCE_DIR']
to_save = f'{PLOTS_DIR}/umap/'
os.makedirs(to_save, exist_ok=True)

In [7]:
adata = ad.read_h5ad('/vol/projects/jnourisa/task_grn_benchmark/resources/datasets_raw/op_perturbation_sc_counts.h5ad', backed='r')

In [10]:
adata.obs['cell_type'].unique()

['B cells', 'T cells CD4+', 'Myeloid cells', 'NK cells', 'T regulatory cells', 'T cells CD8+']
Categories (6, object): ['B cells', 'Myeloid cells', 'NK cells', 'T cells CD4+', 'T cells CD8+', 'T regulatory cells']

In [42]:
# preprocessing could be the issue: check how op was used to be preprocessed
dataset = 'op'
# cell_type='T cells CD8+'
cell_type='CD8T'
feature_type = 'tf_activity'
# adata_sub = retrieve_adata(dataset=dataset, data_type='bulk', cell_type=cell_type)
adata_sub = retrieve_feature_data(dataset=dataset, data_type='bulk', cell_type=cell_type, feature_type=feature_type)
# adata_sub
# adata_n = ad.read_h5ad('/vol/projects/jnourisa/datasets/bulk/op.h5ad')
# adata_n.obs['condition'] = adata_n.obs['perturbation']
# adata_sub = adata_n[adata_n.obs['cell_type']==cell_type,:].copy()
adata_sub

ValueError: File /home/jnourisa/projs/ongoing/hiara//output//features//tf_activity/bulk/op_CD8T.h5ad does not exist

In [36]:
adata_sub.X.max()

5.6363148037962105

In [18]:
adata_n.obs['cell_type'].unique()

['B cells', 'Myeloid cells', 'NK cells', 'T cells CD4+', 'T cells CD8+', 'T regulatory cells']
Categories (6, object): ['B cells', 'Myeloid cells', 'NK cells', 'T cells CD4+', 'T cells CD8+', 'T regulatory cells']

In [41]:
# Define genes expected to respond to ruxolitinib (JAK inhibitor)
if True:
    if False:
        ruxolitinib_target_genes = {
            'JAK-STAT core': ['JAK1', 'JAK2', 'STAT1', 'STAT3', 'STAT5A', 'STAT5B'],
            'Inflammatory cytokines': ['IL6', 'IL1B', 'TNF', 'IFNG', 'IL2'],
            'Downstream targets': ['SOCS1', 'SOCS3', 'BCL2', 'MYC', 'CXCL10'],
            'Inflammatory response': ['CXCL9', 'CXCL10', 'CCL2', 'CCL5'],
            'Transcription factors': ['IRF1', 'IRF8', 'JUN', 'FOS']
        }

        # Flatten the dictionary to get all genes
        all_target_genes = []
        for category, genes in ruxolitinib_target_genes.items():
            all_target_genes.extend(genes)

        # Check which genes are available in the dataset
        
    else:
        all_target_genes = ['TCF7', 'KLF6', 'LEF1', 'GATA3']
    available_genes = [g for g in all_target_genes if g in adata_sub.var_names]
    print(f"Available target genes in dataset: {len(available_genes)}/{len(all_target_genes)}")
    print(f"Genes: {available_genes}")
    # Function to test differential expression for multiple genes
    import scipy.stats as stats
    def test_ruxo_effect(genes_to_test, adata_data, ctr, condition):    
        results = []
        for gene in genes_to_test:
            if gene not in adata_data.var_names:
                continue
            adata_g = adata_data[:, adata_data.var_names == gene]
            expr_cond1 = adata_g[adata_g.obs['condition'] == ctr].X.toarray().flatten()
            expr_cond2 = adata_g[adata_g.obs['condition'] == condition].X.toarray().flatten()
            assert len(expr_cond1) > 0 and len(expr_cond2) > 0, f"No data for gene {gene} in one of the conditions"
            if len(expr_cond1) == 0 or len(expr_cond2) == 0:
                continue
            t_stat, p_value = stats.ttest_ind(expr_cond1, expr_cond2)
            mean_cond1 = expr_cond1.mean()
            mean_cond2 = expr_cond2.mean()
            fold_change = mean_cond2 - mean_cond1
            log2fc = np.log2((mean_cond2 + 1) / (mean_cond1 + 1))
            results.append({
                'gene': gene,
                'mean_control': mean_cond1,
                'mean_ruxo': mean_cond2,
                'fold_change': fold_change,
                'log2_fold_change': log2fc,
                'direction': 'UP' if fold_change > 0 else 'DOWN',
                't_stat': t_stat,
                'p_value': p_value
            })
        df_results = pd.DataFrame(results)
        valid_pvals = df_results['p_value'].notna() & (df_results['p_value'] >= 0) & (df_results['p_value'] <= 1)
        if valid_pvals.sum() > 0:
            df_results['p_adj'] = np.nan
            df_results.loc[valid_pvals, 'p_adj'] = stats.false_discovery_control(
                df_results.loc[valid_pvals, 'p_value'].values
            )
        else:
            df_results['p_adj'] = np.nan
        df_results = df_results.sort_values('p_value')
        return df_results

    if dataset=='CXCL9':
        adata_test = adata_sub.copy()
        # Test RPMI condition
        print("\n" + "="*60)
        print("### RPMI Condition ###")
        ctr = f'24 h RPMI'
        cond = f'24 h RPMI + ruxolitinib'

        results_rpmi = test_ruxo_effect(
            available_genes, adata_test, ctr, cond
        )
        print(f"Comparing: {ctr} vs {cond}")
        print("\nTop 10 most significant:")
        print(results_rpmi.head(10)[['gene', 'log2_fold_change', 'direction', 'p_value', 'p_adj']])

        
        # Test LPS condition
        print("\n### LPS Condition ###")
        ctr = f'24 h LPS'
        cond = f'24 h LPS + ruxolitinib'

        results_lps = test_ruxo_effect(
            available_genes, adata_test, ctr, cond
        )
        print(f"Comparing: {ctr} vs {cond}")
        print("\nTop 10 most significant:")
        print(results_lps.head(10)[['gene', 'log2_fold_change', 'direction', 'p_value', 'p_adj']])

    elif dataset=='op':
        ctr = 'Dimethyl Sulfoxide'
        cond = 'Ruxolitinib'
        results_op = test_ruxo_effect(
            available_genes, adata_sub, ctr, cond
        )

        print(f"Comparing: {ctr} vs {cond}")
        print("\nTop 10 most significant:")
        print(results_op.head(10)[['gene', 'log2_fold_change', 'direction', 'p_value', 'p_adj']])

Available target genes in dataset: 4/4
Genes: ['TCF7', 'KLF6', 'LEF1', 'GATA3']
Comparing: Dimethyl Sulfoxide vs Ruxolitinib

Top 10 most significant:
    gene  log2_fold_change direction   p_value     p_adj
3  GATA3          0.728553      DOWN  0.000271  0.001083
0   TCF7               NaN        UP  0.000567  0.001133
1   KLF6          1.279238      DOWN  0.001774  0.002365
2   LEF1         -0.350286        UP  0.027875  0.027875
